# 02 - Fetch and Process Crime Statistics

Data source: Portal Estadístico de Criminalidad (Ministerio del Interior)

**Download Instructions:**
1. Visit: https://estadisticasdecriminalidad.ses.mir.es/publico/portalestadistico/seriesAnuales
2. Download "Hechos conocidos" by crime type and province → Save as `data/raw/crime_by_type.xlsx`
3. Download "Detenciones" with nationality data → Save as `data/raw/arrests_nationality.xlsx`

**Output:** `data/processed/crime_statistics.json` with crime rates by type, province, and nationality

In [1]:
import requests
import pandas as pd
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, List
from pydantic import BaseModel

In [ ]:
class CrimeRecord(BaseModel):
    year: int
    province: str
    crime_type: str  # Homicidio, Robo, Hurto, etc.
    total_crimes: int
    arrests: int
    nationality: str | None = None  # Española, Extranjera, or None if not available
    crime_rate_per_1000: float | None = None  # Calculated with population
    source_url: str
    fetched_at: str

In [ ]:
def load_crime_by_type(file_path: str) -> pd.DataFrame:
    """
    Load crime statistics by type and province from downloaded Excel/CSV.
    Expected columns: Province, Year, Crime Type, Total
    """
    file_path = Path(file_path)
    
    if not file_path.exists():
        print(f"❌ File not found: {file_path}")
        print(f"Please download from: https://estadisticasdecriminalidad.ses.mir.es/")
        return None
    
    # Try different formats
    if file_path.suffix == '.xlsx':
        df = pd.read_excel(file_path)
    elif file_path.suffix == '.csv':
        # Try different separators
        df = pd.read_csv(file_path, sep=';', encoding='latin-1')
    else:
        print(f"Unsupported format: {file_path.suffix}")
        return None
    
    print(f"✓ Loaded {file_path.name}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    return df

In [ ]:
def load_arrests_by_nationality(file_path: str) -> pd.DataFrame:
    """
    Load arrest statistics with nationality breakdown.
    Expected columns: Province, Year, Nationality, Arrests
    """
    file_path = Path(file_path)
    
    if not file_path.exists():
        print(f"❌ File not found: {file_path}")
        return None
    
    if file_path.suffix == '.xlsx':
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path, sep=';', encoding='latin-1')
    
    print(f"✓ Loaded {file_path.name}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    return df

Testing URL: https://estadisticasdecriminalidad.ses.mir.es/jaxiPx/Tabla.htm?path=/Datos1/&file=01001.px?type=csv


In [ ]:
# Load crime data
raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("Loading crime statistics...")
print("=" * 80)

# Check for downloaded files
crime_file = raw_dir / "crime_by_type.xlsx"
arrests_file = raw_dir / "arrests_nationality.xlsx"

# Alternative CSV names if user downloads CSV instead
crime_csv = raw_dir / "crime_by_type.csv"
arrests_csv = raw_dir / "arrests_nationality.csv"

# Load crime by type
if crime_file.exists():
    df_crime = load_crime_by_type(crime_file)
elif crime_csv.exists():
    df_crime = load_crime_by_type(crime_csv)
else:
    print(f"\\n⚠️  Crime data not found. Please download:")
    print(f"   1. Go to: https://estadisticasdecriminalidad.ses.mir.es/publico/portalestadistico/datos.html?type=jaxi&title=Hechos%20conocidos&path=/Datos1/")
    print(f"   2. Select table with crime types by province")
    print(f"   3. Download as Excel/CSV")
    print(f"   4. Save to: {crime_file}")
    df_crime = None

# Load arrests by nationality  
if arrests_file.exists():
    df_arrests = load_arrests_by_nationality(arrests_file)
elif arrests_csv.exists():
    df_arrests = load_arrests_by_nationality(arrests_csv)
else:
    print(f"\\n⚠️  Arrests data not found. Please download:")
    print(f"   1. Go to: https://estadisticasdecriminalidad.ses.mir.es/publico/portalestadistico/datos.html?type=jaxi&title=Detenciones&path=/Datos3/")
    print(f"   2. Select table with nationality breakdown")
    print(f"   3. Download as Excel/CSV")
    print(f"   4. Save to: {arrests_file}")
    df_arrests = None

In [ ]:
# Inspect the data structure
if df_crime is not None:
    print("\\n" + "=" * 80)
    print("CRIME DATA STRUCTURE")
    print("=" * 80)
    print(df_crime.head())
    print(f"\\nUnique crime types: {df_crime['Crime Type'].nunique() if 'Crime Type' in df_crime.columns else 'Column not found'}")

if df_arrests is not None:
    print("\\n" + "=" * 80)
    print("ARRESTS DATA STRUCTURE")
    print("=" * 80)
    print(df_arrests.head())
    print(f"\\nUnique nationalities: {df_arrests['Nationality'].nunique() if 'Nationality' in df_arrests.columns else 'Column not found'}")

In [ ]:
def process_crime_data(df_crime: pd.DataFrame, df_arrests: pd.DataFrame = None) -> List[Dict]:
    """
    Process and clean crime statistics.
    Normalize column names and merge with arrest data if available.
    """
    if df_crime is None:
        return []
    
    # Make a copy
    df = df_crime.copy()
    
    # Normalize column names (handle different download formats)
    df.columns = df.columns.str.strip().str.lower()
    
    # Common column mappings
    column_mapping = {
        'provincia': 'province',
        'province': 'province',
        'año': 'year',
        'year': 'year',
        'tipo de delito': 'crime_type',
        'crime type': 'crime_type',
        'tipo delito': 'crime_type',
        'infracciones penales': 'crime_type',
        'total': 'total_crimes',
        'hechos conocidos': 'total_crimes',
        'número': 'total_crimes'
    }
    
    df.rename(columns=column_mapping, inplace=True)
    
    # Ensure required columns exist
    required_cols = ['province', 'year', 'crime_type', 'total_crimes']
    missing = [col for col in required_cols if col not in df.columns]
    
    if missing:
        print(f"⚠️  Missing columns: {missing}")
        print(f"Available columns: {list(df.columns)}")
        print("\\nPlease adjust column_mapping in the code or check the downloaded file structure.")
        return []
    
    # Clean data
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df['total_crimes'] = pd.to_numeric(df['total_crimes'], errors='coerce')
    df = df.dropna(subset=['year', 'total_crimes'])
    df['year'] = df['year'].astype(int)
    df['total_crimes'] = df['total_crimes'].astype(int)
    
    # Remove totals/aggregates
    df = df[~df['province'].str.contains('Total|TOTAL|Nacional', case=False, na=False)]
    
    print(f"✓ Processed {len(df)} crime records")
    print(f"  Years: {df['year'].min()} - {df['year'].max()}")
    print(f"  Provinces: {df['province'].nunique()}")
    print(f"  Crime types: {df['crime_type'].nunique()}")
    
    return df

In [ ]:
def export_crime_statistics(df: pd.DataFrame):
    """
    Export processed crime statistics to JSON.
    """
    if df is None or len(df) == 0:
        print("❌ No data to export")
        return
    
    output_dir = Path("../data/processed")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Prepare records
    records = []
    for _, row in df.iterrows():
        record = {
            "year": int(row['year']),
            "province": row['province'],
            "crime_type": row['crime_type'],
            "total_crimes": int(row['total_crimes']),
            "arrests": int(row.get('arrests', 0)) if 'arrests' in row else None,
            "nationality": row.get('nationality', None),
            "source_url": "https://estadisticasdecriminalidad.ses.mir.es/",
            "fetched_at": datetime.now().isoformat()
        }
        records.append(record)
    
    # Export JSON
    output_file = output_dir / "crime_statistics.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    
    print(f"✓ Exported {len(records)} records to {output_file}")
    
    # Also export CSV for backup
    csv_file = output_dir / "crime_statistics.csv"
    df.to_csv(csv_file, index=False, encoding='utf-8')
    print(f"✓ Exported CSV backup to {csv_file}")

In [ ]:
# Process and export
if df_crime is not None:
    df_processed = process_crime_data(df_crime, df_arrests)
    
    if len(df_processed) > 0:
        # Show sample
        print("\\n" + "=" * 80)
        print("SAMPLE OF PROCESSED DATA")
        print("=" * 80)
        print(df_processed.head(10))
        
        # Export
        export_crime_statistics(df_processed)
else:
    print("\\n⚠️  No data to process. Please download the files first.")

## Summary

After running this notebook successfully, you'll have:
- `data/raw/crime_by_type.xlsx` - Raw crime data (gitignored)
- `data/processed/crime_statistics.json` - Clean JSON for the app (committed to git)
- `data/processed/crime_statistics.csv` - CSV backup (gitignored)

**Crime data includes:**
- Crime rates by type (homicide, robbery, theft, assault, etc.)
- Breakdown by province
- Yearly trends
- Nationality of arrests (if available in downloaded data)